# Chapter 5: Schedule Optimization Overview

This notebook walks through the chapter 5 decision chain: parallelism, pipeline, layout, roofline, and autotuning.
The goal is not to benchmark a real kernel here, but to build the same diagnosis habit the chapter text asks for: first decide which layer is limiting the kernel, then decide which optimization is worth trying next.

## 1. Import the shared helpers

We reuse the same lightweight profile model as the scripts so the notebook stays aligned with the chapter text and the command-line demos.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "HANDOFF.md").exists():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from examples.ch05_schedule_optimize_core import classify, get_profiles, next_step
from examples.benchmark_record_template import set_benchmark_record

peak_compute_tflops = 312.0
peak_memory_gbs = 2000.0

print(f"Peak compute: {peak_compute_tflops:.1f} TFLOPS")
print(f"Peak memory:  {peak_memory_gbs:.1f} GB/s")

## 2. Read the roofline-style classification

Each profile below is intentionally simple. The point is to see how the same judge can separate a memory-bound case from a compute-bound one, and then suggest the next optimization layer.

In [ ]:
for profile in get_profiles():
    bottleneck = classify(profile, peak_compute_tflops, peak_memory_gbs)
    print(profile.name)
    print(f"  FLOPs: {profile.flops:.3e}")
    print(f"  Bytes: {profile.bytes_accessed:.3e}")
    print(f"  AI:    {profile.arithmetic_intensity:.2f} FLOP/Byte")
    print(f"  Class: {bottleneck}")
    print(f"  Next:  {next_step(profile, peak_compute_tflops, peak_memory_gbs)}")
    print()

## 3. Map the result to chapter 5 decisions

The notebook is useful if it helps answer the chapter's practical questions:

- Should I change parallelism first, or pipeline first?
- Is this kernel memory-bound or compute-bound?
- Does the current structure already look good enough to stop and validate?
- Is autotuning worth entering now, or should I fix the structure first?

In [ ]:
record = set_benchmark_record(
    scenario="chapter 5 schedule optimization overview",
    operator="analysis",
    platform="NVIDIA",
    target="cuda",
    dtype="float32",
    shape="analysis only",
    baseline="roofline estimate",
    optimization="shared memory / pipeline / layout",
)

for key in ["scenario", "operator", "platform", "target", "dtype", "shape", "baseline", "optimization"]:
    print(f"{key}: {record[key]}")

## 4. What this notebook is for

This notebook is deliberately lightweight. It does not try to be a full performance lab. It exists to make the chapter's decision order concrete:

1. decide the bottleneck class
2. choose the first optimization layer
3. avoid blind parameter search until the structure looks sane